In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

In [2]:
sp = pd.read_csv(r"C:\Users\gaurav\Downloads\spam1.csv", encoding="cp1252")

In [3]:
sp.head(2)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN


In [4]:
sp.isnull().sum()

v1               0
v2               0
Unnamed: 2    6720
Unnamed: 3    6760
Unnamed: 4    6768
dtype: int64

In [5]:
sp.shape

(6776, 5)

In [6]:
sp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6776 entries, 0 to 6775
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          6776 non-null   object
 1   v2          6776 non-null   object
 2   Unnamed: 2  56 non-null     object
 3   Unnamed: 3  16 non-null     object
 4   Unnamed: 4  8 non-null      object
dtypes: object(5)
memory usage: 264.8+ KB


In [7]:
sp = sp.iloc[:,[0,1]] 

In [8]:
sp.head(2)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...


In [9]:
sp = sp.rename(columns = {'v1':'Y','v2':'X'}) # rename the column 

In [10]:
sp.Y = sp.Y.replace({'ham':0,'spam':1})

In [11]:
from sklearn.model_selection import train_test_split
train , test = train_test_split(sp, test_size = .2)

In [12]:
train_x = train.iloc[:,1]
train_y = train.iloc[:,0]

test_x = test.iloc[:,1]
test_y = test.iloc[:,0]

In [13]:
from keras.layers import Dense , SimpleRNN, Embedding
from keras.models import Sequential
from tensorflow.keras.utils import to_categorical

In [14]:
train_y = to_categorical(train_y)

In [15]:
max_num_words = 10000 # how many words do we consider from the entre corpus
seq_len = 50 # how many tokens( words to consider in each document/message)
embedding_size = 100 # vector length of each words
from tensorflow.keras.preprocessing.text import Tokenizer # assign a number to each word
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [16]:
# padding in CNN
# pad_sequences -- adding extra layers of zero

In [17]:
tokenizer = Tokenizer(num_words=max_num_words)
tokenizer.fit_on_texts(sp.X)
train_x = tokenizer.texts_to_sequences(train_x)
train_x = pad_sequences(train_x , maxlen = seq_len)

In [18]:
train_x
# train_x is a list containg 5420 list

array([[   0,    0,    0, ..., 3490, 2320, 6607],
       [   0,    0,    0, ..., 3616,  125,  184],
       [   0,    0,    0, ...,    6,  976,  481],
       ...,
       [   0,    0,    0, ...,  144,   50,   90],
       [   0,    0,    0, ...,  108,   62, 1006],
       [   0,    0,    0, ...,  201, 1878,  487]],
      shape=(5420, 50), dtype=int32)

In [19]:
# for model building each input should be of same length

In [20]:
test_x = tokenizer.texts_to_sequences(test_x)
test_x = pad_sequences(test_x , maxlen = seq_len)

In [21]:
test_x

array([[   0,    0,    0, ...,   10,  323, 1094],
       [   0,    0,    0, ...,   14,  113,  141],
       [   0,    0,    0, ...,  100, 1551, 1003],
       ...,
       [   0,    0,    0, ...,  608,  433,   15],
       [   0,    0,    0, ...,  934,  228,  225],
       [   0,    0,    0, ...,  564,  210,  221]],
      shape=(1356, 50), dtype=int32)

In [61]:
model = Sequential()  # initialize the network
model.add(Embedding(input_dim=max_num_words, output_dim=embedding_size))
model.add(SimpleRNN(10))
model.add(Dense(2, activation='softmax'))  # O/p: 2 because you have 2 levels in target variable
from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate=0.01)

# Correct parameter: 'optimizer' (singular), passing the custom 'adam' object
model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])

In [62]:
model.fit(train_x, train_y, epochs = 5, batch_size = 63, validation_split =.2)

Epoch 1/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9532 - loss: 0.1327 - val_accuracy: 0.9825 - val_loss: 0.0684
Epoch 2/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9954 - loss: 0.0181 - val_accuracy: 0.9843 - val_loss: 0.0567
Epoch 3/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9993 - loss: 0.0025 - val_accuracy: 0.9899 - val_loss: 0.0516
Epoch 4/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 1.0000 - loss: 7.2460e-04 - val_accuracy: 0.9899 - val_loss: 0.0582
Epoch 5/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 3.3837e-04 - val_accuracy: 0.9889 - val_loss: 0.0639


In [63]:
pred = model.predict(test_x)
pred_cate = pred.argmax(axis=1)
tab = confusion_matrix(test_y,pred_cate)
tab.diagonal().sum()/ tab.sum()

43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


np.float64(0.9918879056047197)

In [27]:
tab

array([[1169,    5],
       [  22,  160]])

In [64]:
model.save("12_oct_RNN_spam1_NN.h5")